In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os, sys
SRC = "/kaggle/input/datasets/jaganadhg/code-vase-v2/src"     # adjust slug if different
sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
import literank; print("literank OK ->", literank.__file__)


literank OK -> /kaggle/input/datasets/jaganadhg/code-vase-v2/src/literank/__init__.py


In [3]:
#!cp -r /kaggle/input/datasets/jaganadhg/code-base/* .

In [4]:
!pytest -m "not integration" -q


no tests ran in 0.00s


In [5]:
!python -c "import literank, torch; print('literank', literank.__version__, '| cuda', torch.cuda.is_available())"

literank 0.1.0 | cuda True


## Resume

## Successful

In [6]:
# !python -m literank.cli train --scorer lite --proj-dim 768 \
#   --subset-size 500000 --max-steps 50000 --batch-size 64 \
#   --eval-every 2000 --patience 3 --keep-last 3 \
#   --checkpoint-dir /kaggle/working/ckpt_lite_big \
#   --resume /kaggle/input/datasets/jaganadhg/last-checkoint/ckpt_step44000.pt --device cuda

In [7]:
### Large Data Run

import os, subprocess
COMMON = ["--subset-size", "2000000",   # caps at the full train split
        "--max-steps", "40000", "--batch-size", "64",
        "--eval-every", "2000", "--patience", "3", "--keep-last", "3", "--device", "cuda"]

def launch(gpu, scorer):
  rundir = f"/kaggle/working/run_{scorer}_full"; os.makedirs(rundir, exist_ok=True)
  env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
  cmd = ["python", "-m", "literank.cli", "train", "--scorer", scorer,
         "--checkpoint-dir", f"/kaggle/working/ckpt_{scorer}_full", *COMMON]
  return subprocess.Popen(cmd, cwd=rundir, env=env,
                          stdout=open(f"/kaggle/working/{scorer}_full.log", "w"),
                          stderr=subprocess.STDOUT)

procs = {"lite": launch(0, "lite"), "maxsim": launch(1, "maxsim")}
print("launched:", {k: p.pid for k, p in procs.items()})

launched: {'lite': 51, 'maxsim': 52}


In [8]:
import glob, os, re, torch
from datasets import load_dataset
from literank.config import ModelConfig, DataConfig
from literank.model import Ranker
from literank.checkpoint import load_checkpoint
from literank.rerank import rerank
from literank.evaluate import mrr_at_k, ndcg_at_k

N_DEV = 2000
SEARCH_DIRS = sorted(glob.glob("/kaggle/working/ckpt_*"))   # add "/kaggle/input/<dataset>/..." dirs if needed

# Build the dev slice once (streaming; keep queries that have at least one relevant passage).
dcfg = DataConfig()
stream = load_dataset(dcfg.dataset_name, dcfg.dataset_config, split="validation", streaming=True)
dev = []
for rec in stream:
  passages = rec["passages"]["passage_text"]; labels = rec["passages"]["is_selected"]
  if passages and sum(labels) > 0:
      dev.append((rec["query"], passages, labels))
  if len(dev) >= N_DEV:
      break
print(f"dev queries: {len(dev)}")

def pick_checkpoint(d):
  best = os.path.join(d, "best.pt")
  if os.path.exists(best):
      return best
  steps = glob.glob(os.path.join(d, "ckpt_step*.pt"))
  return max(steps, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p)).group(1))) if steps else None

@torch.no_grad()
def evaluate(ckpt_path):
  blob = torch.load(ckpt_path, map_location="cpu", weights_only=False)
  cfg = ModelConfig(**blob["config"])                 # use the checkpoint's own architecture
  ranker = Ranker(cfg).to("cuda"); load_checkpoint(ckpt_path, ranker, map_location="cuda"); ranker.eval()
  ranked = []
  for query, passages, labels in dev:
      doc_embs, doc_masks = ranker.encoder.encode(passages, cfg.doc_len)
      q_emb, q_mask = ranker.encoder.encode([query], cfg.query_len)
      order = rerank(ranker.scorer, q_emb, q_mask, doc_embs, doc_masks, device="cuda")
      ranked.append([labels[i] for i in order])
  return cfg.scorer, mrr_at_k(ranked, 10), ndcg_at_k(ranked, 10)

print(f"{'scorer':8} {'dir':26} {'ckpt':16} {'MRR@10':>8} {'nDCG@10':>8}")
found = False
for d in SEARCH_DIRS:
  ck = pick_checkpoint(d)
  if ck is None:
      continue
  found = True
  scorer, mrr, ndcg = evaluate(ck)
  print(f"{scorer:8} {os.path.basename(d):26} {os.path.basename(ck):16} {mrr:8.4f} {ndcg:8.4f}")
if not found:
  print("No checkpoints under /kaggle/working/ckpt_*. If they are in an attached Dataset, "
        "add its /kaggle/input/<name> dir(s) to SEARCH_DIRS above.")
print("\nNote: scores use each query's own candidate passages (not BM25 top-1000); only the "
    "relative LITE-vs-MaxSim comparison is meaningful, not the absolute vs the paper's 0.393.")


dev queries: 2000
scorer   dir                        ckpt               MRR@10  nDCG@10
No checkpoints under /kaggle/working/ckpt_*. If they are in an attached Dataset, add its /kaggle/input/<name> dir(s) to SEARCH_DIRS above.

Note: scores use each query's own candidate passages (not BM25 top-1000); only the relative LITE-vs-MaxSim comparison is meaningful, not the absolute vs the paper's 0.393.


In [9]:
# LITE_DIR   = "/kaggle/working/ckpt_lite_big"     # or "/kaggle/working/ckpt_lite"
# MAXSIM_DIR = "/kaggle/working/ckpt_maxsim_big"   # or "/kaggle/working/ckpt_maxsim"; set None to skip
# N_DEV = 2000

# import glob, os, re, torch
# from datasets import load_dataset
# from literank.config import ModelConfig, DataConfig
# from literank.model import Ranker
# from literank.checkpoint import load_checkpoint
# from literank.rerank import rerank
# from literank.evaluate import mrr_at_k, ndcg_at_k

# def pick_checkpoint(ckpt_dir):
#   """Prefer best.pt (early-stopping peak); else the highest-numbered ckpt_step*.pt."""
#   best = os.path.join(ckpt_dir, "best.pt")
#   if os.path.exists(best):
#       return best
#   paths = glob.glob(os.path.join(ckpt_dir, "ckpt_step*.pt"))
#   if not paths:
#       raise FileNotFoundError(f"no checkpoints in {ckpt_dir}")
#   return max(paths, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p)).group(1)))

# dcfg = DataConfig()
# dev = load_dataset(dcfg.dataset_name, dcfg.dataset_config, split="validation")
# dev = dev.select(range(min(N_DEV, len(dev))))

# @torch.no_grad()
# def evaluate(scorer, ckpt_dir):
#   ckpt = pick_checkpoint(ckpt_dir)
#   ranker = Ranker(ModelConfig(scorer=scorer, proj_dim=768)).to("cuda")
#   load_checkpoint(ckpt, ranker, map_location="cuda")
#   ranker.eval()
#   ranked = []
#   for rec in dev:
#       passages = rec["passages"]["passage_text"]
#       labels = rec["passages"]["is_selected"]
#       if not passages or sum(labels) == 0:
#           continue
#       doc_embs, doc_masks = ranker.encoder.encode(passages, ranker.cfg.doc_len)
#       q_emb, q_mask = ranker.encoder.encode([rec["query"]], ranker.cfg.query_len)
#       order = rerank(ranker.scorer, q_emb, q_mask, doc_embs, doc_masks, device="cuda")
#       ranked.append([labels[i] for i in order])
#   return os.path.basename(ckpt), mrr_at_k(ranked, 10), ndcg_at_k(ranked, 10)

# print(f"{'model':8} {'ckpt':16} {'MRR@10':>8} {'nDCG@10':>8}")
# for name, scorer, d in [("LITE", "lite", LITE_DIR), ("MaxSim", "maxsim", MAXSIM_DIR)]:
#   if not d:
#       continue
#   ck, mrr, ndcg = evaluate(scorer, d)
#   print(f"{name:8} {ck:16} {mrr:8.4f} {ndcg:8.4f}")



In [10]:
# import os, glob, re, subprocess

# COMMON = ["--subset-size", "500000", "--max-steps", "50000", "--batch-size", "64",
#         "--eval-every", "2000", "--patience", "3", "--keep-last", "3", "--device", "cuda"]

# def latest_ckpt(d):
#   paths = glob.glob(f"{d}/ckpt_step*.pt")
#   return max(paths, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", p).group(1))) if paths else None

# def resume(gpu, scorer):
#   ckdir  = f"/kaggle/working/ckpt_{scorer}_big"
#   rundir = f"/kaggle/working/run_{scorer}_big"          # same cwd -> reuses cached teacher_scores.json
#   os.makedirs(rundir, exist_ok=True)
#   env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
#   cmd = ["python", "-m", "literank.cli", "train", "--scorer", scorer,
#          "--checkpoint-dir", ckdir, *COMMON]
#   last = latest_ckpt(ckdir)
#   if last:
#       cmd += ["--resume", last]; print(f"{scorer}: resuming from {last}")
#   else:
#       print(f"{scorer}: no checkpoint found, starting fresh")
#   logf = open(f"/kaggle/working/{scorer}_big.log", "a")   # append, keep history
#   return subprocess.Popen(cmd, cwd=rundir, env=env, stdout=logf, stderr=subprocess.STDOUT)

# procs = {"lite": resume(0, "lite"), "maxsim": resume(1, "maxsim")}
# print("resumed:", {k: p.pid for k, p in procs.items()})


In [11]:
  # import os, subprocess

  # # Scale-up run: more DATA (the real lever) + early stopping to peak on dev MRR.
  # COMMON = ["--subset-size", "500000",   # ~300k triplets (5x more diverse data, not just more epochs)
  #           "--max-steps", "80000",      # ~10 epochs over the bigger set (cap)
  #           "--batch-size", "64",
  #           "--eval-every", "2000",      # periodic dev MRR@10 ...
  #           "--patience", "3",           # ... stop after 3 non-improving evals; saves best.pt
  #           "--keep-last", "3",
  #           "--device", "cuda"]

  # def launch(gpu, scorer):
  #     rundir = f"/kaggle/working/run_{scorer}_big"      # separate cwd -> isolates teacher_scores.json
  #     os.makedirs(rundir, exist_ok=True)
  #     env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
  #     cmd = ["python", "-m", "literank.cli", "train", "--scorer", scorer,
  #            "--checkpoint-dir", f"/kaggle/working/ckpt_{scorer}_big", *COMMON]
  #     logf = open(f"/kaggle/working/{scorer}_big.log", "w")
  #     return subprocess.Popen(cmd, cwd=rundir, env=env, stdout=logf, stderr=subprocess.STDOUT)

  # procs = {"lite": launch(0, "lite"), "maxsim": launch(1, "maxsim")}
  # print("launched:", {k: p.pid for k, p in procs.items()})

In [12]:
# import os, subprocess

# COMMON = ["--subset-size", "100000", "--max-steps", "20000",
#         "--batch-size", "64",          # <-- was 16; uses the idle memory + raises util
#         "--keep-last", "3", "--device", "cuda"]

# def launch(gpu, scorer):
#   rundir = f"/kaggle/working/run_{scorer}"
#   os.makedirs(rundir, exist_ok=True)
#   env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
#   cmd = ["python", "-m", "literank.cli", "train", "--scorer", scorer,
#          "--checkpoint-dir", f"/kaggle/working/ckpt_{scorer}", *COMMON]
#   logf = open(f"/kaggle/working/{scorer}.log", "w")
#   return subprocess.Popen(cmd, cwd=rundir, env=env, stdout=logf, stderr=subprocess.STDOUT)

# procs = {"lite": launch(0, "lite"), "maxsim": launch(1, "maxsim")}
# print("launched:", {k: p.pid for k, p in procs.items()})


In [13]:
!echo "=== LITE ===";   tail -n 4 /kaggle/working/lite.log
!echo "=== MAXSIM ==="; tail -n 4 /kaggle/working/maxsim.log
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv


=== LITE ===
tail: cannot open '/kaggle/working/lite.log' for reading: No such file or directory
=== MAXSIM ===
tail: cannot open '/kaggle/working/maxsim.log' for reading: No such file or directory
index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 0 MiB
1, 0 %, 0 MiB


In [14]:
# for k, p in procs.items():
#   p.wait(); print(k, "exit code", p.returncode)

In [15]:
# import glob, os
# for d in sorted(glob.glob("/jaganadhg/literank-repro-v1/ckpt_*")):
#   pts = sorted(glob.glob(d + "/*.pt"))
#   print(d, "->", [os.path.basename(p) for p in pts[-3:]] or "EMPTY")

In [16]:
# # Small-LITE storage ablation: compare cached embedding bytes at proj_dim=768 vs proj_dim=128.
# # `encode_and_cache` returns the size (bytes) of the saved cache file -- the "storage lever"
# # from the paper's Small-LITE variant (projecting to a smaller d' shrinks the offline doc cache).
# import torch
# from literank.config import ModelConfig
# from literank.encoder import DualEncoder

# sample_docs = [
#     "The quick brown fox jumps over the lazy dog.",
#     "LITE is a lightweight late-interaction re-ranker for document retrieval.",
#     "Kaggle provides free GPU compute for training and inference.",
# ] * 20  # small stand-in batch; the dev-doc cache built below uses the real MS MARCO dev split

# from literank.encode_cache import encode_and_cache

# cfg_768 = ModelConfig(proj_dim=768)
# enc_768 = DualEncoder(cfg_768).to("cuda")
# bytes_768 = encode_and_cache(enc_768, sample_docs, cfg_768.doc_len,
#                               "/kaggle/working/ablation_proj768.pt", batch_size=32)

# cfg_128 = ModelConfig(proj_dim=128)
# enc_128 = DualEncoder(cfg_128).to("cuda")
# bytes_128 = encode_and_cache(enc_128, sample_docs, cfg_128.doc_len,
#                               "/kaggle/working/ablation_proj128.pt", batch_size=32)

# print(f"proj_dim=768 cache size: {bytes_768:,} bytes")
# print(f"proj_dim=128 cache size: {bytes_128:,} bytes")
# print(f"storage ratio (128/768): {bytes_128 / bytes_768:.3f}")
# assert bytes_128 < bytes_768, "Small-LITE (proj_dim=128) should use less storage than proj_dim=768"


In [17]:
# # Encode a dev slice of MS MARCO, rerank candidates per query against the trained LITE
# # checkpoint (in memory, no per-query disk round-trip), and compute MRR@10 / nDCG@10.
# import torch
# from datasets import load_dataset
# from literank.config import ModelConfig, DataConfig
# from literank.encoder import DualEncoder
# from literank.model import Ranker
# from literank.checkpoint import load_checkpoint
# from literank.rerank import rerank
# from literank.evaluate import mrr_at_k, ndcg_at_k

# dcfg = DataConfig()
# mcfg = ModelConfig(scorer="lite", proj_dim=768)

# dev = load_dataset(dcfg.dataset_name, dcfg.dataset_config, split="validation")
# dev = dev.select(range(min(dcfg.num_dev_queries, len(dev))))

# # Load the trained LITE checkpoint produced by the training cell above.
# ranker = Ranker(mcfg).to("cuda")
# import glob, os, re

# def latest_checkpoint(ckpt_dir):
#     # Checkpoints are written every `checkpoint_every` steps and a Kaggle session is
#     # often interrupted before `max_steps`, so pick the highest-numbered checkpoint
#     # rather than hard-coding a step count.
#     paths = glob.glob(os.path.join(ckpt_dir, "ckpt_step*.pt"))
#     if not paths:
#         raise FileNotFoundError(f"no checkpoints in {ckpt_dir}")
#     return max(paths, key=lambda p: int(re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p)).group(1)))

# latest_ckpt = latest_checkpoint("/kaggle/working/ckpt_lite_v3")
# load_checkpoint(latest_ckpt, ranker, map_location="cuda")
# ranker.eval()

# all_relevances = []
# for rec in dev:
#     query = rec["query"]
#     passages = rec["passages"]["passage_text"]
#     is_selected = rec["passages"]["is_selected"]
#     if not passages:
#         continue

#     # Encode this query's candidate passages and the query itself directly in memory
#     # (no encode_and_cache/load_embeddings disk round-trip per query) and rerank.
#     with torch.no_grad():
#         doc_embs, doc_masks = ranker.encoder.encode(passages, mcfg.doc_len)
#         q_emb, q_mask = ranker.encoder.encode([query], mcfg.query_len)
#     order = rerank(ranker.scorer, q_emb, q_mask, doc_embs, doc_masks, device="cuda")

#     relevances = [is_selected[i] for i in order]
#     all_relevances.append(relevances)

# mrr10 = mrr_at_k(all_relevances, k=10)
# ndcg10 = ndcg_at_k(all_relevances, k=10)
# print(f"Dev MRR@10:  {mrr10:.4f}")
# print(f"Dev nDCG@10: {ndcg10:.4f}")
# print("Note: the paper reports MRR@10=0.393 on the full MS MARCO dev set at full training "
#       "scale. This subset/Kaggle run is expected to score lower -- it is a qualitative "
#       "reproduction of the LITE architecture and training recipe, not a leaderboard result.")
